# GTSRB Research: Milestone 2 — ResNet-18 Baseline & Robustness Benchmark

This notebook runs the **standard black-box CNN baseline (ResNet-18)** under the frozen **Master Dataset v1** contract:
1. **Architecture:** Pretrained ResNet-18 with 43-class head and Bag-of-Tricks zero-$\gamma$ initialization.
2. **Optimization:** SGD (momentum 0.9, Nesterov), Linear Warmup (3 eps) + Cosine Annealing decay, No-Bias-Decay rule.
3. **Preprocessing:** ROI crop with 10% context margin $\to$ aspect-ratio preserving letterbox $\to$ $64 \times 64$ zero-padded canvas.
4. **Splits:** Track-stratified group split (`train.csv`: 31,378, `val.csv`: 7,830, `test.csv`: 12,630 held-out).
5. **Evaluation:** Clean accuracy, Macro-F1, ECE, and 15 held-out corruptions $\times$ 5 severities.

In [ ]:
# --- Step 1: Environment Setup & GPU Verification ---
from google.colab import drive
drive.mount('/content/drive')

# Copy archive from Drive and unzip
!cp "/content/drive/MyDrive/ML_Project_Data.zip" /content/
!unzip -q /content/ML_Project_Data.zip -d /content/ML_Project

%cd /content/ML_Project

import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
!nvidia-smi

In [ ]:
# --- Step 2: Data Pipeline Sanity Check & Visual Grid ---
import matplotlib.pyplot as plt
import numpy as np
from src.dataset import GTSRBDataset, IMAGENET_MEAN, IMAGENET_STD

train_ds = GTSRBDataset('data/master_v1/metadata/train.csv', 'german', target_size=64)
print(f"Train samples: {len(train_ds):,}")

# Plot 8 sample signs with concept annotations
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, ax in enumerate(axes.flat):
    item = train_ds[idx * 150]
    # Unnormalize tensor for display
    img_np = item['image'].cpu().numpy().transpose(1, 2, 0)
    img_np = (img_np * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)
    
    ax.imshow(img_np)
    ax.set_title(f"{item['class_name']}\nShape: {item['shape']} | Symbol: {item['symbol_content']}", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# --- Step 3: Train ResNet-18 Baseline (Colab T4 GPU) ---
# 25 epochs takes ~4-5 minutes on T4 with Mixed Precision (AMP)
!python train_baseline.py \
    --data-root german \
    --meta-dir data/master_v1/metadata \
    --output-dir checkpoints/baseline_resnet18 \
    --seed 42 \
    --epochs 25 \
    --batch-size 128 \
    --lr 0.05 \
    --num-workers 2

In [ ]:
# --- Step 4: Plot Training & Validation Curves ---
import json
import pandas as pd

with open('checkpoints/baseline_resnet18/training_history.json', 'r') as f:
    history = json.load(f)

epochs_df = pd.DataFrame(history['epochs'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epochs_df['epoch'], epochs_df['train_loss'], label='Train Loss', color='blue', lw=2)
ax1.plot(epochs_df['epoch'], epochs_df['val_loss'], label='Val Loss', color='red', lw=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Cross-Entropy Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy & Macro-F1
ax2.plot(epochs_df['epoch'], epochs_df['val_acc'] * 100, label='Val Accuracy (%)', color='green', lw=2)
ax2.plot(epochs_df['epoch'], epochs_df['val_macro_f1'] * 100, label='Val Macro-F1 (%)', color='purple', lw=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score (%)')
ax2.set_title('Validation Performance')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Step 5: Clean Test & Robustness Benchmark (15 Corruptions x 5 Severities) ---
!python evaluate_robustness.py \
    --checkpoint checkpoints/baseline_resnet18/best_baseline_resnet18.pt \
    --data-root german \
    --meta-path data/master_v1/metadata/test.csv \
    --output-dir reports/baseline_resnet18 \
    --batch-size 128

In [ ]:
# --- Step 6: Display Robustness Matrix & Backup Results to Drive ---
import pandas as pd

matrix_df = pd.read_csv('reports/baseline_resnet18/robustness_matrix.csv')
display(matrix_df)

# Copy checkpoints and reports back to Google Drive
!mkdir -p "/content/drive/MyDrive/ML_Project_Results/baseline_resnet18"
!cp -r checkpoints/baseline_resnet18/* "/content/drive/MyDrive/ML_Project_Results/baseline_resnet18/"
!cp -r reports/baseline_resnet18/* "/content/drive/MyDrive/ML_Project_Results/baseline_resnet18/"
print("\nAll results backed up to Google Drive successfully!")